# Stage Evaluation Walkthrough

Step-by-step notebook for building Value Stream Stage ground truth and running a small stage prediction/evaluation smoke test.

Important: prediction input must use only IDMT ticket content. Theme Business Needs, Theme summaries, verified stages, child issue summaries, and `gt_by_value_stream` are ground-truth/evaluation data only.

## 1. Setup Repo Path

In [ ]:
from __future__ import annotations

import asyncio
import json
import os
import sys
from pathlib import Path
from pprint import pprint

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'src' / 'vs_app').exists():
    REPO = REPO.parent

if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))
if str(REPO / 'scripts') not in sys.path:
    sys.path.insert(0, str(REPO / 'scripts'))

print('repo root:', REPO)

## 2. Configure Ticket and Catalog

By default this notebook loads the local JSON catalog at `data/value_stream_stage_map.json`. Set `STAGE_CATALOG_SOURCE=azure` only when you intentionally want to test against the Azure value-stream index.

In [ ]:
TICKET_ID = 'IDMT-19761'

# Default source is local JSON: data/value_stream_stage_map.json.
# Set STAGE_CATALOG_SOURCE=azure only when testing against the Azure value-stream index.
STAGE_CATALOG_SOURCE = os.getenv("STAGE_CATALOG_SOURCE", "json")
STAGE_CATALOG_PATH = Path(
    os.getenv("STAGE_CATALOG_PATH", REPO / "data" / "value_stream_stage_map.json")
)
FETCH_CHILD_ISSUES = True

print('ticket:', TICKET_ID)
print('catalog source:', STAGE_CATALOG_SOURCE)
print('catalog path:', STAGE_CATALOG_PATH)

## 3. Load Stage Catalog

In [ ]:
from vs_app.modules.stages.stage_catalog import (
    get_allowed_stages,
    get_value_stream_catalog_entry,
    load_stage_catalog,
    summarize_stage_catalog,
)

catalog = load_stage_catalog(
    path=STAGE_CATALOG_PATH,
    source=STAGE_CATALOG_SOURCE,
)

print('value streams in catalog:', len(catalog))
for row in summarize_stage_catalog(catalog, limit=10):
    print(row)

## 4. Fetch IDMT Ticket

In [ ]:
from build_stage_ground_truth import JiraApiClient

jira_base_url = os.getenv('JIRA_BASE_URL', '').strip()
jira_token = os.getenv('JIRA_TOKEN', '').strip()
if not jira_base_url or not jira_token:
    raise RuntimeError('Set JIRA_BASE_URL and JIRA_TOKEN before running Jira cells.')

jira_ctx = JiraApiClient(base_url=jira_base_url, token=jira_token, verify_ssl=False)
jira_client = await jira_ctx.__aenter__()

idmt_issue = await jira_client.get_issue(
    TICKET_ID,
    fields=['summary', 'description', 'issuetype', 'issuelinks', 'attachment'],
    expand=True,
)
idmt_fields = idmt_issue.get('fields') or {}
print('summary:', idmt_fields.get('summary'))
print('issue type:', (idmt_fields.get('issuetype') or {}).get('name'))
print('issue links:', len(idmt_fields.get('issuelinks') or []))

## 5. Inspect Linked Theme Refs

In [ ]:
from vs_app.modules.stages.stage_ground_truth import find_linked_theme_issues

theme_refs = find_linked_theme_issues(idmt_issue)
pprint(theme_refs)

## 6. Build GT for One Ticket

In [ ]:
from vs_app.modules.stages.stage_ground_truth import build_ticket_stage_ground_truth

ticket_gt = await build_ticket_stage_ground_truth(
    ticket_key=TICKET_ID,
    jira_client=jira_client,
    catalog=catalog,
    fetch_child_issues=FETCH_CHILD_ISSUES,
)

print('idmt_key:', ticket_gt['idmt_key'])
print('idmt_summary:', ticket_gt['idmt_summary'])
print('idmt_description chars:', len(ticket_gt.get('idmt_description') or ''))
print('linked themes:', len(ticket_gt.get('linked_themes') or []))
print('gt_by_value_stream:')
pprint(ticket_gt.get('gt_by_value_stream'))

## 7. Inspect Verified and Unresolved Stages

In [ ]:
for theme in ticket_gt.get("linked_themes") or []:
    print("\n" + "=" * 100)
    print("Theme:", theme.get("theme_key"), theme.get("theme_summary"))
    print("Business VS:", theme.get("business_value_stream"))
    print("Allowed stages:", theme.get("allowed_stages"))

    print("\nBusiness Needs mentions debug only:")
    pprint(theme.get("business_needs_mentions_debug_only"))

    print("\nChild issue lookup:")
    pprint(theme.get("child_issue_lookup"))

    print("\nChild issues fetched:")
    pprint(theme.get("child_issues"))

    print("\nVerified stages:")
    pprint(theme.get("verified_stages"))

    print("\nUnresolved stage mentions:")
    pprint(theme.get("unresolved_stage_mentions"))

    print("\nWarnings:")
    pprint(theme.get("warnings"))

## 7.1 Focused Check for GROUP-22223

In [ ]:
for theme in ticket_gt.get("linked_themes") or []:
    if theme.get("theme_key") == "GROUP-22223":
        print("GT for GROUP-22223")
        pprint(theme.get("child_issue_lookup"))
        pprint(theme.get("child_issues"))
        pprint(theme.get("child_issue_mentions_debug"))
        pprint(theme.get("canonicalization_debug"))
        pprint(theme.get("verified_stages"))
        print("gt_by_value_stream[Manage Utilization Management Program]")
        pprint((ticket_gt.get("gt_by_value_stream") or {}).get("Manage Utilization Management Program"))

## 7.2 Child Issue Stage Mapping Table

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

rows = []

for theme in ticket_gt.get("linked_themes") or []:
    value_stream = (theme.get("business_value_stream") or {}).get("name")
    allowed_stages = theme.get("allowed_stages") or []

    child_by_key = {
        child.get("key"): child
        for child in theme.get("child_issues") or []
    }

    canonical_by_child_raw = {}
    for item in theme.get("canonicalization_debug") or []:
        key = (
            item.get("child_key"),
            item.get("raw_stage"),
            item.get("source_text"),
        )
        canonical_by_child_raw[key] = item

    for mention in theme.get("child_issue_mentions_debug") or []:
        child_key = mention.get("child_key")
        child = child_by_key.get(child_key) or {}

        lookup_key = (
            child_key,
            mention.get("raw_stage"),
            mention.get("source_text"),
        )
        mapped = canonical_by_child_raw.get(lookup_key) or {}

        rows.append(
            {
                "theme_key": theme.get("theme_key"),
                "value_stream": value_stream,
                "child_key": child_key,
                "child_issue_type": child.get("issue_type"),
                "child_status": child.get("status"),
                "child_summary": child.get("summary") or mention.get("source_text"),
                "cleaned_raw_stage": mention.get("raw_stage"),
                "canonical_stage": mapped.get("canonical"),
                "match_method": mapped.get("match_method"),
                "confidence": mapped.get("confidence"),
                "warnings": "; ".join(mapped.get("warnings") or []),
                "allowed_stages": " | ".join(allowed_stages),
            }
        )

if pd is None:
    if rows:
        pprint(rows)
    else:
        print("No child issue stage mentions found. Check child_issue_lookup for JQL attempts.")
else:
    child_stage_df = pd.DataFrame(rows)
    if child_stage_df.empty:
        print("No child issue stage mentions found. Check child_issue_lookup for JQL attempts.")
    else:
        display(child_stage_df)

In [ ]:
focus_theme_key = "GROUP-22223"

focus_rows = [
    row
    for row in rows
    if row.get("theme_key") == focus_theme_key
]

if pd is None:
    if focus_rows:
        pprint(focus_rows)
    else:
        print(f"No child mapping rows found for {focus_theme_key}")
        for theme in ticket_gt.get("linked_themes") or []:
            if theme.get("theme_key") == focus_theme_key:
                print("Child lookup:")
                pprint(theme.get("child_issue_lookup"))
                print("Child issues:")
                pprint(theme.get("child_issues"))
                print("Child issue mentions debug:")
                pprint(theme.get("child_issue_mentions_debug"))
                print("Canonicalization debug:")
                pprint(theme.get("canonicalization_debug"))
else:
    focus_df = pd.DataFrame(focus_rows)
    if focus_df.empty:
        print(f"No child mapping rows found for {focus_theme_key}")
        for theme in ticket_gt.get("linked_themes") or []:
            if theme.get("theme_key") == focus_theme_key:
                print("Child lookup:")
                pprint(theme.get("child_issue_lookup"))
                print("Child issues:")
                pprint(theme.get("child_issues"))
                print("Child issue mentions debug:")
                pprint(theme.get("child_issue_mentions_debug"))
                print("Canonicalization debug:")
                pprint(theme.get("canonicalization_debug"))
    else:
        display(focus_df)

## 7.3 Final GT by Value Stream

In [ ]:
gt_rows = []

for value_stream, stages in (ticket_gt.get("gt_by_value_stream") or {}).items():
    gt_rows.append(
        {
            "ticket_id": ticket_gt.get("idmt_key"),
            "value_stream": value_stream,
            "gt_stage_count": len(stages or []),
            "gt_stages": " | ".join(stages or []),
        }
    )

if pd is None:
    pprint(gt_rows)
else:
    gt_df = pd.DataFrame(gt_rows)
    display(gt_df)

## 7.4 Allowed Stage Catalog Check

In [ ]:
catalog_rows = []

for theme in ticket_gt.get("linked_themes") or []:
    value_stream = (theme.get("business_value_stream") or {}).get("name")
    catalog_rows.append(
        {
            "theme_key": theme.get("theme_key"),
            "value_stream": value_stream,
            "allowed_stage_count": len(theme.get("allowed_stages") or []),
            "allowed_stages": " | ".join(theme.get("allowed_stages") or []),
        }
    )

if pd is None:
    pprint(catalog_rows)
else:
    catalog_df = pd.DataFrame(catalog_rows)
    display(catalog_df)

## 8. Save Single-Ticket GT JSON

In [ ]:
out_dir = REPO / 'output' / 'stage_eval' / 'notebook'
out_dir.mkdir(parents=True, exist_ok=True)
gt_payload = {
    'source': 'jira',
    'stage_catalog_source': STAGE_CATALOG_SOURCE,
    'tickets': {TICKET_ID: ticket_gt},
}
gt_path = out_dir / 'stage_ground_truth_single.json'
gt_path.write_text(json.dumps(gt_payload, indent=2, ensure_ascii=False), encoding='utf-8')
print('wrote:', gt_path)

## 9. Build Prediction Input from IDMT Only

This cell intentionally uses only `idmt_summary` and `idmt_description`. It must not read Theme Business Needs, Theme summaries, verified stages, child issues, or `gt_by_value_stream`.

In [ ]:
prediction_input = '\n\n'.join(
    part
    for part in (
        ticket_gt.get('idmt_summary') or '',
        ticket_gt.get('idmt_description') or '',
    )
    if part.strip()
)

print('prediction_input chars:', len(prediction_input))
print(prediction_input[:1200])

## 10. Run Stage Prediction Smoke Test

In [ ]:
from evaluate_stage_batch import _make_generation_service
from vs_app.modules.stages.stage_selector import predict_value_stream_stages

llm = _make_generation_service()
predictions = {}
for value_stream_name in (ticket_gt.get('gt_by_value_stream') or {}):
    catalog_entry = get_value_stream_catalog_entry(value_stream_name, catalog) or {}
    allowed = get_allowed_stages(value_stream_name, catalog)
    predictions[value_stream_name] = predict_value_stream_stages(
        idea_card_text=prediction_input,
        value_stream_name=value_stream_name,
        allowed_stages=allowed,
        value_stream_description=catalog_entry.get('description') or '',
        llm=llm,
    )

pprint(predictions)

## 11. Compute Metrics

In [ ]:
from vs_app.modules.stages.stage_metrics import evaluate_stage_predictions

rows = []
for value_stream_name, gt_stages in (ticket_gt.get('gt_by_value_stream') or {}).items():
    predicted = [
        item.get('stage')
        for item in (predictions.get(value_stream_name) or {}).get('predicted_stages') or []
        if item.get('stage')
    ]
    rows.append(
        {
            'ticket_id': TICKET_ID,
            'value_stream_name': value_stream_name,
            'gt_stages': gt_stages,
            'predicted_stages': predicted,
        }
    )

metrics = evaluate_stage_predictions(rows)
pprint(metrics.get('summary'))
pprint(metrics.get('rows'))

## 12. Batch Commands

```powershell
$env:JIRA_BASE_URL="https://jira.fyiblue.com"
$env:JIRA_TOKEN="..."

py -3 scripts/build_stage_ground_truth.py `
  --ticket-ids-file data/valid_tickets.txt `
  --stage-catalog-source json `
  --stage-catalog data/value_stream_stage_map.json `
  --fetch-child-issues `
  --output output/stage_eval/stage_ground_truth.json

py -3 scripts/evaluate_stage_batch.py `
  --ticket-ids-file data/valid_tickets.txt `
  --stage-ground-truth output/stage_eval/stage_ground_truth.json `
  --stage-catalog-source json `
  --stage-catalog data/value_stream_stage_map.json `
  --vs-source ground_truth `
  --output-dir output/stage_eval/run_gt_vs
```


## 13. Teardown

In [ ]:
await jira_ctx.__aexit__(None, None, None)
print('jira client closed')